In [ ]:
!pip install feedparser beautifulsoup4 lxml --quiet

import feedparser
import pandas as pd
import numpy as np
import time
import hashlib
import re
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from datetime import datetime


In [ ]:
# ==============================
# PROJECT SETTINGS
# ==============================

SUBREDDIT = "TaylorSwift"

SEARCH_TERMS = [
    "Taylor Swift",
    "Eras Tour",
    "album",
    "The Tortured Poets Department",
    "Midnights",
    "Reputation",
    "Grammy",
    "ticket",
    "music video",
    "performance"
]

# RSS collection limits
MAX_POSTS_TOTAL = 250
MAX_COMMENTS_PER_POST = 30

# Be polite to Reddit RSS
REQUEST_PAUSE_SECONDS = 1.0

USER_AGENT = "MSc Web and Social Media Analytics Coursework - Taylor Swift Reddit RSS"

In [ ]:
# ==============================
# HELPER FUNCTIONS
# ==============================

def html_to_text(html):
    """
    Convert RSS HTML summaries into readable plain text.
    """
    if html is None:
        return ""
    soup = BeautifulSoup(str(html), "lxml")
    text = soup.get_text(" ", strip=True)
    return text


def clean_text_basic(text):
    """
    Basic cleaning for raw RSS text.
    Deeper preprocessing will be done later.
    """
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def anonymise_author(author):
    """
    Anonymise Reddit usernames for ethical reporting.
    """
    if author is None or str(author).strip() == "":
        author = "unknown"
    return hashlib.sha256(str(author).encode()).hexdigest()[:12]


def parse_rss(url):
    """
    Read an RSS feed with a user-agent.
    """
    feed = feedparser.parse(
        url,
        request_headers={"User-Agent": USER_AGENT}
    )
    return feed


def safe_get(entry, field, default=""):
    """
    Safely get a value from a feedparser entry.
    """
    return entry.get(field, default)


def get_entry_datetime(entry):
    """
    Extract published date if available.
    """
    published = safe_get(entry, "published", "")
    try:
        return pd.to_datetime(published)
    except:
        return pd.NaT

In [ ]:
# ==============================
# CREATE RSS FEED URLS
# ==============================

feed_urls = []

# Main subreddit feeds
feed_urls.append(f"https://www.reddit.com/r/{SUBREDDIT}/new/.rss")
feed_urls.append(f"https://www.reddit.com/r/{SUBREDDIT}/hot/.rss")
feed_urls.append(f"https://www.reddit.com/r/{SUBREDDIT}/top/.rss?t=year")

# Search feeds inside the subreddit
for term in SEARCH_TERMS:
    encoded_term = quote_plus(term)
    feed_urls.append(
        f"https://www.reddit.com/r/{SUBREDDIT}/search.rss?q={encoded_term}&restrict_sr=1&sort=new&t=year"
    )

# Add old.reddit fallback versions
old_feed_urls = [url.replace("https://www.reddit.com", "https://old.reddit.com") for url in feed_urls]

all_feed_urls = feed_urls + old_feed_urls

print("Number of RSS feed URLs prepared:", len(all_feed_urls))
for url in all_feed_urls[:5]:
    print(url)

In [ ]:
# ==============================
# COLLECT POSTS FROM RSS
# ==============================

post_rows = []
seen_links = set()

for feed_url in all_feed_urls:
    print("Reading feed:", feed_url)

    feed = parse_rss(feed_url)

    if len(feed.entries) == 0:
        print("No entries found for this feed.")
        time.sleep(REQUEST_PAUSE_SECONDS)
        continue

    for entry in feed.entries:
        link = safe_get(entry, "link", "")

        if link in seen_links:
            continue

        seen_links.add(link)

        title = clean_text_basic(safe_get(entry, "title", ""))
        summary = clean_text_basic(html_to_text(safe_get(entry, "summary", "")))
        author = safe_get(entry, "author", "unknown")

        post_rows.append({
            "source_platform": "Reddit",
            "subreddit": SUBREDDIT,
            "post_title": title,
            "post_summary": summary,
            "post_author_id": anonymise_author(author),
            "post_author_raw": author,
            "post_link": link,
            "post_published": safe_get(entry, "published", ""),
            "post_datetime": get_entry_datetime(entry),
            "rss_feed_url": feed_url
        })

        if len(post_rows) >= MAX_POSTS_TOTAL:
            break

    time.sleep(REQUEST_PAUSE_SECONDS)

    if len(post_rows) >= MAX_POSTS_TOTAL:
        break

posts_df = pd.DataFrame(post_rows)

print("Number of unique Reddit posts collected:", len(posts_df))
posts_df.head()

In [ ]:
# ==============================
# CREATE COMMENT RSS URLS
# ==============================

def make_comment_rss_candidates(post_link):
    """
    Create possible RSS versions of a Reddit comment-thread URL.
    Reddit URL formats can vary, so we try multiple candidates.
    """
    if not isinstance(post_link, str) or post_link.strip() == "":
        return []

    base = post_link.strip().split("?")[0].rstrip("/")

    candidates = [
        base + ".rss",
        base + "/.rss",
        base.replace("https://www.reddit.com", "https://old.reddit.com") + ".rss",
        base.replace("https://www.reddit.com", "https://old.reddit.com") + "/.rss"
    ]

    return list(dict.fromkeys(candidates))


comment_rss_urls = []

for link in posts_df["post_link"].dropna().unique():
    for candidate in make_comment_rss_candidates(link):
        comment_rss_urls.append(candidate)

print("Number of possible comment RSS URLs:", len(comment_rss_urls))
comment_rss_urls[:5]

In [ ]:
# ==============================
# COLLECT COMMENTS FROM COMMENT RSS FEEDS
# ==============================

comment_rows = []
seen_comment_links = set()

for idx, post_row in posts_df.iterrows():
    post_link = post_row["post_link"]
    candidates = make_comment_rss_candidates(post_link)

    comments_for_this_post = 0

    for comment_feed_url in candidates:
        feed = parse_rss(comment_feed_url)

        if len(feed.entries) == 0:
            continue

        for entry in feed.entries:
            comment_link = safe_get(entry, "link", "")

            if comment_link in seen_comment_links:
                continue

            seen_comment_links.add(comment_link)

            title = clean_text_basic(safe_get(entry, "title", ""))
            summary = clean_text_basic(html_to_text(safe_get(entry, "summary", "")))
            author = safe_get(entry, "author", "unknown")

            # Skip entries that are too short to be useful
            if len(summary) < 10 and len(title) < 10:
                continue

            comment_rows.append({
                "source_platform": "Reddit",
                "subreddit": SUBREDDIT,
                "post_title": post_row["post_title"],
                "post_link": post_link,
                "comment_title": title,
                "comment_body": summary,
                "comment_author_id": anonymise_author(author),
                "comment_author_raw": author,
                "comment_link": comment_link,
                "comment_published": safe_get(entry, "published", ""),
                "comment_datetime": get_entry_datetime(entry),
                "comment_rss_url": comment_feed_url
            })

            comments_for_this_post += 1

            if comments_for_this_post >= MAX_COMMENTS_PER_POST:
                break

        if comments_for_this_post >= MAX_COMMENTS_PER_POST:
            break

        time.sleep(REQUEST_PAUSE_SECONDS)

    if idx % 10 == 0:
        print(f"Processed {idx + 1}/{len(posts_df)} posts. Comments collected so far: {len(comment_rows)}")

comments_df = pd.DataFrame(comment_rows)

print("Number of Reddit comments collected from RSS:", len(comments_df))
comments_df.head()

In [ ]:
# ==============================
# CREATE COMBINED RAW DATASET
# ==============================

if len(comments_df) > 0:
    raw_df = comments_df.copy()
    raw_df["text_for_analysis"] = raw_df["comment_body"]
    raw_df["content_type"] = "comment"
else:
    raw_df = posts_df.copy()
    raw_df["text_for_analysis"] = raw_df["post_title"] + " " + raw_df["post_summary"]
    raw_df["content_type"] = "post"

print("Raw dataset shape:", raw_df.shape)
print("Content type used:", raw_df["content_type"].iloc[0])
raw_df.head()

In [ ]:
# ==============================
# BASIC DATASET CHECKS
# ==============================

print("Number of rows in raw dataset:", len(raw_df))
print("Number of posts collected:", len(posts_df))
print("Number of comments collected:", len(comments_df))

print("\nDate range:")
date_col = "comment_datetime" if "comment_datetime" in raw_df.columns else "post_datetime"
print("Earliest:", raw_df[date_col].min())
print("Latest:", raw_df[date_col].max())

print("\nSample text:")
display(raw_df[["text_for_analysis"]].head(10))

# ==============================
# SAVE RAW DATASETS
# ==============================

posts_df.to_csv("taylor_swift_reddit_rss_posts_raw.csv", index=False)
comments_df.to_csv("taylor_swift_reddit_rss_comments_raw.csv", index=False)
raw_df.to_csv("taylor_swift_reddit_rss_raw_dataset.csv", index=False)

print("Saved files:")
print("1. taylor_swift_reddit_rss_posts_raw.csv")
print("2. taylor_swift_reddit_rss_comments_raw.csv")
print("3. taylor_swift_reddit_rss_raw_dataset.csv")